# BKinD on CalMS21 (noise runs) — Colab

Same training as the Kaggle notebook (`bkind_calms21_kaggle.ipynb`): all the work is done by
`scripts/kaggle_bkind.py`, cloned fresh from GitHub each session, so change code there and push.
Colab has one GPU, so run **one** run per session. One run takes ~10–11 h on a T4 (faster on L4/A100).

**Setup (once):**
1. The frames are downloaded from the Kaggle dataset `sleekeagle/calms21-train` with kagglehub
   (the same data the Kaggle notebook uses). Google Drive holds only the run outputs.
2. *Runtime → Change runtime type →* a **GPU** (T4 or better).
3. If the GitHub repo is private: Secrets panel (key icon), add `GITHUB_TOKEN` with read access.

Checkpoints, logs and samples are saved to `MyDrive/noisykeypoints/runs/<name>/` after every epoch.
**If Colab disconnects:** reconnect and *Run all* again with the same `RUNS`; training continues from
the last finished epoch (at most one epoch, ~45 min, is lost).


In [ ]:
# ---- Settings ----
# Other noise types (1 seed). Order: all types at level 2 first (SSIM 0.6, the one level every type
# has, so they compare directly with sensor noise L2), then the severe level of each type.
RUNS = [dict(name='CalMS21_ll_L2_s1', seed=1, overrides=dict(noise=dict(type='low_light', level=2, seed=0)))]
# Queue (one per session):
#   dict(name='CalMS21_sp_L2_s1',   seed=1, overrides=dict(noise=dict(type='speckle',     level=2, seed=0)))
#   dict(name='CalMS21_blur_L2_s1', seed=1, overrides=dict(noise=dict(type='motion_blur', level=2, seed=0)))
#   dict(name='CalMS21_h264_L2_s1', seed=1, overrides=dict(noise=dict(type='h264',        level=2, seed=0)))
#   dict(name='CalMS21_ll_L4_s1',   seed=1, overrides=dict(noise=dict(type='low_light',   level=4, seed=0)))
#   dict(name='CalMS21_sp_L4_s1',   seed=1, overrides=dict(noise=dict(type='speckle',     level=4, seed=0)))
#   dict(name='CalMS21_blur_L1_s1', seed=1, overrides=dict(noise=dict(type='motion_blur', level=1, seed=0)))
#   dict(name='CalMS21_h264_L1_s1', seed=1, overrides=dict(noise=dict(type='h264',        level=1, seed=0)))
REPO_URL = 'https://github.com/sleekEagle/noisykeypoints.git'
MAX_HOURS = 11.5        # stop cleanly (between epochs) before this many hours into the session

import os, sys, time, subprocess
T0 = time.time()
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/noisykeypoints'

# The CalMS21 frames, from the Kaggle dataset (cached for the rest of this Colab session).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'kagglehub'], check=True)
import kagglehub
DATA_IN = kagglehub.dataset_download('sleekeagle/calms21-train')
print('CalMS21 download:', DATA_IN)

# Get the code (uses the GITHUB_TOKEN secret if the repo is private) and load the helper.
CODE = '/content/noisykeypoints'
url = REPO_URL
try:
    from google.colab import userdata
    url = REPO_URL.replace('https://', f'https://{userdata.get("GITHUB_TOKEN")}@')
except Exception:
    print('no GITHUB_TOKEN secret; cloning as a public repo')
if os.path.exists(CODE):
    subprocess.run(['git', '-C', CODE, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', url, CODE], check=True)
sys.path.insert(0, f'{CODE}/scripts')
import kaggle_bkind


In [ ]:
runs = kaggle_bkind.run('calms21', RUNS, max_hours=MAX_HOURS, t0=T0,
                        inp=DATA_IN, data='/content/data', out=f'{DRIVE}/runs')
